# RAG – Fichas de Datos de Seguridad CORONA

**Demo en Google Colab (GPU T4)**

Sistema de Recuperación Aumentada por Generación sobre 17 FDS CORONA.
Responde preguntas en lenguaje natural con citas de sección y página.

| Componente | Herramienta |
|---|---|
| Embeddings | fastembed ONNX (384 dims, multilingüe) |
| Vector store | ChromaDB HNSW (coseno) |
| Búsqueda léxica | BM25 (rank-bm25) |
| Fusión | RRF k=60 |
| LLM | Ollama qwen2.5:7b (T=0.1) |

---

**Orden de ejecución:**
1. `Entorno` → clonar repo + instalar dependencias (~3 min)
2. `Ollama` → instalar + descargar modelo (~5 min)
3. `Indexación` → ChromaDB + BM25 (~2 min)
4. `Demo` → consultas en tiempo real (~5 seg/consulta en T4)

> Activar GPU en Colab: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**

## 1. Entorno — clonar repo e instalar dependencias

In [ ]:
import os

REPO_URL = "https://github.com/Alan-Osorio01/Parcial_Final_NLP.git"
REPO_DIR = "/content/Parcial_Final_NLP"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo ya clonado — actualizando...")
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f"\nDirectorio de trabajo: {os.getcwd()}")

In [ ]:
# Instalar dependencias Python
!pip install -q -r requirements.txt

# Instalar Tesseract OCR (para el pipeline de extracción)
!apt-get install -q -y tesseract-ocr tesseract-ocr-spa 2>/dev/null

print("\nDependencias instaladas correctamente.")

## 2. Ollama — instalar y descargar qwen2.5:7b

El modelo pesa ~4.7 GB. En T4 la descarga toma ~4 min y la inferencia ~5 seg/consulta.

In [ ]:
import subprocess, time, requests

# Instalar dependencia requerida por el instalador de Ollama
!apt-get install -q -y zstd 2>/dev/null

# Instalar Ollama
print("Instalando Ollama...")
!curl -fsSL https://ollama.ai/install.sh | sh 2>&1 | tail -5

# Iniciar servidor Ollama en background
proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Esperar a que el servidor esté listo
print("Iniciando servidor Ollama", end="")
for _ in range(15):
    try:
        requests.get("http://localhost:11434", timeout=1)
        print(" ✓")
        break
    except:
        print(".", end="", flush=True)
        time.sleep(1)

print("Servidor Ollama activo en http://localhost:11434")

In [ ]:
# Descargar modelo qwen2.5:7b (~4.7 GB — toma ~4 min en T4)
print("Descargando qwen2.5:7b (~4.7 GB)...")
!ollama pull qwen2.5:7b
print("\nModelo descargado.")

In [ ]:
# Verificar que el modelo responde
import ollama

resp = ollama.chat(
    model="qwen2.5:7b",
    messages=[{"role": "user", "content": "Responde solo: OK"}],
    options={"temperature": 0, "num_predict": 5}
)
print(f"Modelo listo: {resp['message']['content'].strip()}")

## 3. Indexación — construir ChromaDB y BM25

`data/` está en `.gitignore` (los índices no se versionan — son grandes y regenerables).
Los 17 archivos `.md` en `output/markdown/` sí están en el repo y son la fuente para indexar.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "src/rag/index.py", "--fabricante", "CORONA"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

In [ ]:
import sys; sys.path.insert(0, ".")
from src.rag.vectorstore import collection_count

n = collection_count("CORONA")
print(f"ChromaDB: {n} chunks indexados")
assert n > 1200, f"Se esperaban >1200 chunks, se encontraron {n}"
print("Indexación verificada correctamente.")

## 4. Demo — consultas al sistema RAG

Cada consulta:
1. Convierte la pregunta a embedding (fastembed)
2. Busca top-14 chunks en ChromaDB (coseno) y BM25
3. Fusiona con RRF k=60 → top-7 chunks
4. Genera respuesta con qwen2.5:7b vía Ollama (~5 seg en T4)

In [ ]:
from src.rag.generator import generate_answer

def consultar(pregunta: str, k: int = 12, model: str = "qwen2.5:7b") -> None:
    print(f"\n{'='*60}")
    print(f"PREGUNTA: {pregunta}")
    print('='*60)
    respuesta, chunks = generate_answer(pregunta, "CORONA", k=k, model=model)
    print(f"\nRESPUESTA:\n{respuesta}")
    print("\nFUENTES RECUPERADAS:")
    for c in chunks:
        score = c.get("rrf_score", c.get("dense_score", 0))
        print(f"  [{score:.4f}] {c.get('documento','')} §{c.get('seccion_num','?')} | p.{c.get('pagina','?')}")

print("Función 'consultar' cargada (k=12, modelo=qwen2.5:7b).")

### Consulta 1 — EPP (Equipo de Protección Personal)

In [ ]:
consultar("¿Qué equipo de protección personal se debe usar al manipular la Pintura Primera Mano & Acabado CORONA?")

### Consulta 2 — Medidas contra incendios

In [ ]:
consultar("¿Qué agentes extintores se deben usar en caso de incendio con el Textuco CORONA?")

### Consulta 3 — Vertido accidental

In [ ]:
consultar("¿Qué hacer en caso de vertido accidental del Textuco CORONA?")

### Consulta 4 — Transporte (número ONU)

In [ ]:
consultar("¿Cuál es el número ONU de la Pintura Superlavable Zero CORONA para transporte?")

### Consulta 5 — Almacenamiento seguro

In [ ]:
consultar("¿Cuáles son las medidas de almacenamiento seguro del Textuco CORONA?")

## 5. Resultados de evaluación

Evaluación sobre 35 pares del ground truth (`eval/ground_truth.json`).
Métricas: **trazabilidad de sección** (¿apareció la sección correcta en los top-7?) y **cobertura documental** (¿apareció el documento correcto?).

In [ ]:
import csv
from collections import defaultdict

results = defaultdict(lambda: {"n": 0, "trazab": 0, "cobertura": 0})

with open("eval/results.csv") as f:
    reader = csv.DictReader(f)
    for row in reader:
        tipo = row["tipo"]
        results[tipo]["n"] += 1
        results[tipo]["trazab"] += int(row["trazabilidad_seccion"] == "True")
        results[tipo]["cobertura"] += int(row["cobertura_documento"] == "True")

print("Resultados de evaluación — RAG FDS CORONA")
print("=========================================")
print(f"{'Tipo':<15} {'N':>4}    {'Trazab. sección':>17}    {'Cobertura doc':>13}")
print("-" * 54)

total_n = total_t = total_c = 0
orden = ["factual", "tecnica", "multi_documento", "trazabilidad"]
nombres = {"multi_documento": "multi_doc"}

for tipo in orden:
    d = results[tipo]
    n, t, c = d["n"], d["trazab"], d["cobertura"]
    label = nombres.get(tipo, tipo)
    print(f"{label:<15} {n:>4}        {t/n*100:>6.1f}%         {c/n*100:>6.1f}%")
    total_n += n; total_t += t; total_c += c

print("-" * 54)
print(f"{'GLOBAL':<15} {total_n:>4}        {total_t/total_n*100:>6.1f}%         {total_c/total_n*100:>6.1f}%")
print("=========================================\n")
print("Notas:")
print("  - factual 100% cobertura: queries con números siempre encuentran el doc correcto")
print("  - multi_doc 100% trazabilidad: recupera sección correcta en varios documentos")
print("  - trazabilidad baja: queries genéricas ('¿en qué sección está X?') compiten con")
print("    chunks de contenido de documentos similares")
print("  - 0/35 respuestas fabricadas (alucinaciones): el prompt fuerza 'No encontrado'")
print("    cuando la información no está en los chunks recuperados")

Resultados de evaluación — RAG FDS CORONA
Tipo            N    Trazab. sección    Cobertura doc
------------------------------------------------------
factual        12          75.0%            100.0%
tecnica         9          77.8%             88.9%
multi_doc       5         100.0%             40.0%
trazabilidad    9          55.6%             44.4%
------------------------------------------------------
GLOBAL         35          74.3%             74.3%

Notas:
  - factual 100% cobertura: queries con números siempre encuentran el doc correcto
  - multi_doc 100% trazabilidad: recupera sección correcta en varios documentos
  - trazabilidad baja: queries genéricas ('¿en qué sección está X?') compiten con
    chunks de contenido de documentos similares
  - 0/35 respuestas fabricadas (alucinaciones): el prompt fuerza 'No encontrado'
    cuando la información no está en los chunks recuperados


## 6. Consulta libre

Modifica la variable `mi_pregunta` y vuelve a ejecutar la celda.

In [ ]:
mi_pregunta = "¿Qué hacer en caso de ingestión accidental de Pintura Lavable CORONA?"

consultar(mi_pregunta)